# Temperature-Dependent Body-Diode ANN

In [ ]:
# Import the measured-data loader and the paper-style body-diode ANN.
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

from DeviceData import load_body_diode_characteristics
from body_diode_hybrid import BodyDiodeNN, train_ibd_network, predict_ibd, save_ibd_pkl, load_ibd_pkl


## Load the three measured junction temperatures

In [ ]:
# Read Vds and Ids from every Vgs curve in the three body-diode folders.
BASE_DIR = Path.cwd()
TEMPLATE_DIR = BASE_DIR / "Template"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
df = load_body_diode_characteristics(TEMPLATE_DIR)
print(df.groupby(["T_C", "Vgs"]).size())
print("Total body-diode points =", len(df))

Vgs = torch.tensor(df["Vgs"].to_numpy(np.float32)).reshape(-1, 1)
Vds = torch.tensor(df["Vds"].to_numpy(np.float32)).reshape(-1, 1)
T_C = torch.tensor(df["T_C"].to_numpy(np.float32)).reshape(-1, 1)
Ibd = torch.tensor(df["Ibd"].to_numpy(np.float32)).reshape(-1, 1)
MODEL_FILE = BASE_DIR / "Ibd_NN.pkl"


## Train Eq. (29) with Tj added as the temperature input

In [ ]:
# Keep two hidden layers with six neurons and add Tj only at the input layer.
EPOCHS = 5000
LEARNING_RATE = 1.0e-4
BATCH_SIZE = 128
VAL_FRACTION = 0.20
SEED = 42

model, norm, history = train_ibd_network(
    Vgs,
    Vds,
    T_C,
    Ibd,
    device=device,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    val_fraction=VAL_FRACTION,
    seed=SEED,
    print_every=250,
)
print("Best validation loss =", history.best_val_loss)


In [ ]:
# Save the trained temperature-aware body-diode neural network as a structured .pkl file.
save_ibd_pkl(
    MODEL_FILE,
    model,
    norm,
    training_metadata={
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "validation_fraction": VAL_FRACTION,
        "seed": SEED,
        "best_validation_loss": float(history.best_val_loss),
        "number_of_samples": int(len(df)),
        "temperatures_C": sorted(df["T_C"].unique().tolist()),
    },
)
print("Saved =", MODEL_FILE)


## Static-characteristic accuracy

In [ ]:
# Predict every measured point and report the static-current fit error.
Ibd_pred = predict_ibd(
    model,
    norm,
    Vgs.reshape(-1),
    Vds.reshape(-1),
    T_C.reshape(-1),
    device=device,
).cpu().numpy().reshape(-1)
Ibd_true = Ibd.numpy().reshape(-1)
error = Ibd_pred - Ibd_true
print("RMSE =", float(np.sqrt(np.mean(error**2))), "A")
print("MAE =", float(np.mean(np.abs(error))), "A")


In [ ]:
# Plot measured and ANN body-diode curves at all three temperatures.
for temp_c in [-55.0, 25.0, 150.0]:
    plt.figure(figsize=(7.2, 5.2))
    part_t = df[np.isclose(df["T_C"], temp_c)]
    for vgs_value in sorted(part_t["Vgs"].unique()):
        measured = part_t[np.isclose(part_t["Vgs"], vgs_value)].sort_values("Vds")
        plt.scatter(measured["Vds"], measured["Ibd"], s=20)
        Vds_curve = np.linspace(measured["Vds"].min(), 0.0, 300)
        Vgs_curve = np.full_like(Vds_curve, vgs_value)
        T_curve = np.full_like(Vds_curve, temp_c)
        Ibd_curve = predict_ibd(model, norm, Vgs_curve, Vds_curve, T_curve, device=device).cpu().numpy().reshape(-1)
        plt.plot(Vds_curve, Ibd_curve, label=rf"$V_{{GS}}={vgs_value:g}$ V")
    plt.xlabel(r"$V_{DS}$ (V)")
    plt.ylabel(r"$I_{bd}$ (A)")
    plt.title(rf"Body-Diode Characteristics at $T_J={temp_c:g}^\circ$C")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Verify that the saved .pkl file reloads independently.
model_loaded, norm_loaded, checkpoint = load_ibd_pkl(MODEL_FILE, device=device)
print(checkpoint["model_name"])
print(checkpoint["architecture"])
print(checkpoint["input_names"])


In [ ]:
# Plot the training and validation losses.
plt.figure(figsize=(6, 4))
plt.semilogy(history.train_loss, label="Training")
plt.semilogy(history.val_loss, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Normalized MSE")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
